# T5.1 Estructuras de datos lineales: introducción y conceptos básicos

## 1. Introducción

### ¿Qué es una estructura de datos?

Una **estructura de datos** es una forma de organizar y representar un conjunto de datos elementales con el objetivo de permitir su manipulación de forma sencilla y eficiente.

Una estructura de datos representa un **tipo de datos** (clase/objeto) que ofrece una **interfaz de métodos** para realizar operaciones sobre los elementos que almacena.

> **Importante**: La **complejidad temporal** de dichas operaciones depende críticamente de la forma en la que los elementos están organizados internamente.


<div align="center">
    <img src="fig/eds.png" alt="Clasificación de los tipos de datos más comunes" style="max-width:600px"/>
</div>

### Estructuras de datos lineales (EDL)

Las estructuras de datos lineales son aquellas cuyos elementos están formados por **secuencias**:

$$d_0, d_1, \ldots, d_{n-1}, \quad n \geq 0$$

donde los $d_i$ son típicamente datos del mismo tipo y sobre las que se pueden hacer operaciones de:
- **Consulta** de elementos,
- **Inserción** de nuevos elementos,
- **Eliminación** de elementos existentes.

Según la **política de manipulación** de los datos, se distinguen tres EDLs principales:

- **Pila** (*Stack*)
   + Política LIFO (*Last In, First Out*): solo se accede al último elemento insertado (tope)
- **Cola** (*Queue*)
   + Política FIFO (*First In, First Out*): se inserta por un extremo y se extrae por el otro.
- **Lista** (*List*): 
   + Acceso y modificación en cualquier posición.

## 2. Representación de secuencias

Las EDLs típicamente se basan en **dos aproximaciones diferentes** de representar secuencias de datos en memoria:

1. **Almacenamiento contiguo** mediante arrays.
2. **Almacenamiento disperso** mediante nodos enlazados.

Por simplicidad, consideraremos estructuras de datos lineales cuyos elementos son de tipo **entero**, aunque en Python, al ser un lenguaje de tipado dinámico, nuestras implementaciones serán fácilmente generalizables a cualquier tipo de dato.

### 2.1 Representación con almacenamiento contiguo (arrays)

Una manera directa de representar una secuencia consiste en disponer sus elementos de forma **contigua en memoria**, como ocurre con los **arrays** de lenguajes como C/C++ o Java. 

> En Python no existen arrays. En su lugar, usaremos listas, obviando por completo su interfaz pública (métodos de la clase `list`) y sus detalles de implementación interna en C como array dinámico de punteros a objetos (ver [`struct PyListObject`](https://github.com/python/cpython/blob/3.15/Include/cpython/listobject.h#L22)). 

**Características:**
- **Acceso directo** a cualquier elemento por su posición.
- **Talla limitada** a la longitud del array.
    + Si eventualmente se necesita almacenar más elementos que la capacidad máxima del array, se debe crear un nuevo array de mayor capacidad y copiar los elementos existentes al nuevo array.
- La **inserción/eliminación** de un dato en la posición $i$-ésima exige **desplazar** en memoria los elementos posteriores (proceso de compactación del array).
- **Consumo de memoria fijo** con independencia del número de elementos realmente almacenados.


<div align="center">
    <img src="fig/array.png" alt="Almacenamiento contiguo" style="max-width:550px"/>
</div>

### 2.2 Representación con almacenamiento disperso (nodos enlazados)

Otra forma alternativa consiste en disponer los elementos de forma **dispersa en memoria** mediante **nodos** que se conectan explícitamente entre sí.

Un **nodo** es una estructura elemental que almacena:
- Un **dato** (`data`), y
- Un **enlace** o **referencia** (`next`) a la dirección de memoria del siguiente nodo en la secuencia (como mínimo).

**Características:**
- El **acceso** a los datos es **indirecto** por posición: requiere seguir enlaces desde el inicio hasta llegar a la posición deseada.
- La **talla** de la secuencia **no está limitada**. Simplemente requiere la creación de un nuevo nodo y enlazarlo adecuadamente.
- **Consumo de memoria variable**, dependiente del número de elementos almacenados (con el sobrecoste de almacenar los punteros a otros nodos).
- La **inserción/eliminación** de elementos solo implica modificar el enlace del nodo precedente, si bien requiere localizar previamente el punto de inserción.

<div align="center">
    <img src="fig/linked.png" alt="Almacenamiento disperso (nodos enlazados)" style="max-width:600px"/>
</div>

### 2.3 Comparativa

Vamos a comparar ambas aproximaciones, a rasgos generales, en cuanto a coste temporal de operaciones y uso de memoria.

Considerando la longitud $n$ de la secuencia $d_0 \dots d_{n-1}$ como la talla del problema:

|  | Memoria contigua (array) | Memoria dispersa (nodos enlazados) |
|---|---|---|
| Acceso por posición | $\Theta(1)$ — directo | $\Omega(1) \cap O(n)$ — secuencial |
| Inserción/eliminación | $\Omega(1) \cap O(n)$ — desplazamiento | $\Omega(1) \cap O(n)$ — localización + modificación de enlaces |
| Memoria | Fija (posible desperdicio) | Variable (sobrecoste por enlaces) |
| Talla máxima | Limitada (o con redimensionamiento) | Sin límite explícito |

En los próximos cuadernos profundizaremos en la representación con nodos enlazados y la implementaremos en Python, antes de abordar cada EDL (Pila, Cola, Lista) con ambas representaciones.

## 3. Gestión de memoria

La gestión de estructuras de datos implica necesariamente gestión de memoria:
- Se reserva memoria para poder almacenar nuevos datos.
- Se libera memoria cuando dicho espacio previamente reservado ya no es necesario. 

En lenguajes como C++, la creación de nodos en memoria dinámica (heap) se realiza con `new`, y su destrucción (liberación de memoria) con `delete`. El programador es responsable de gestionar la memoria manualmente.

```C++
// C++:  
NodeInt* n = new NodeInt(5);

// Hacer cosas con n... y finalmente, cuando el nodo ya no es necesario:

delete n; // liberamos memoria
```

En Python, los objetos se crean simplemente instanciando la clase, y la liberación de memoria la gestiona automáticamente el _garbage collector_ (recolector de basura), cuando los objetos quedan desreferenciados (contador de referencias toma el valor 0).

Python ofrece la sentencia `del` que permite eliminar referencias a objetos de cualquier tipo (incluso funciones, clases, módulos, etc). 

Si `x` es un nombre de Python que apunta a un objeto cualquiera, la instrucción `del x` elimina dicho nombre `x` del ámbito actual (función). Esto hace decrementar inmediatamente el contador de referencias del objeto apuntado por `x`, sin tener que esperar a que el flujo de ejecución salga del ámbito en el que se declaró o se importó `x`. 

`del` también se puede usar para eliminar elementos de listas y de diccionarios, lo que conlleva eliminar sus correspondientes referencias. P.e.: `del x[1]` o `del x["key"]`.

```python
# Python (equivalente):
n = Node(5)       # Se crea el nodo

# Hacer cosas con n... y finalmente, cuando no es necesario:

del n           
```

En la mayoria de casos no es necesario usar `del`. Lo usaremos solo si es estrictamente necesario.